# Round 2 Algorithmic Trading Strategy

This notebook documents the strategy in `trader.py` for `ASH_COATED_OSMIUM` and `INTARIAN_PEPPER_ROOT`. The goal is maximum expected XIREC profit with explicit controls against regime breaks, stale books, and overfit model selection.


## Executive thesis

The active strategy replays at **248673.0 XIRECs** across days -1, 0, and 1, versus **246681.0 XIRECs** from the archived round 1 parameters. The parameter delta is deliberately small: buy pepper root at an 8-XIREC trend edge, and make osmium crossing slightly more responsive with `fair_alpha = 0.12`, `imbalance_weight = 0.8`, and `take_edge = 0.0`.

The largest edge is structural: `INTARIAN_PEPPER_ROOT` behaves like a nearly deterministic upward-drifting claim. `ASH_COATED_OSMIUM` is stationary around 10,000 with strong one-step mean reversion, so it is traded as an inventory-limited market-making and mean-reversion book.


## Market Access Fee bid

`trader.py` bids **825 XIRECs** for extra market access. This is a measured middle bid: it is far above the public example/default-style `15`, but it does not sacrifice a large fraction of the round's expected PnL if accepted.

A local sensitivity check scaled visible quote volume from 80% to 100% and from 100% to 125%. The extra 25% book access was worth roughly 700-800 XIRECs per final-day equivalent in the crossing-only replay, mostly from additional osmium mean-reversion fills. Passive fill upside is not captured in that simple replay, so 825 is a defensible bid for acceptance probability without turning the blind auction into the main risk of the strategy.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data' / 'round2'
diagnostics = json.loads((ROOT / 'logs' / 'round2_diagnostics.json').read_text())
diagnostics['deterministic_backtest']['combined_pnl']


## Data regime

Rows with `mid_price == 0` have empty visible top-of-book data and are excluded from slope and training calculations. The relevant price process summary is:

| day | product | first | last | slope / 100 | resid sigma | ret ac1 | empty rows |
| --- | --- | --- | --- | --- | --- | --- | --- |
| -1 | ASH_COATED_OSMIUM | 9991.0 | 10002.0 | -0.000053 | 4.464 | -0.506 | 15 |
| -1 | INTARIAN_PEPPER_ROOT | 11001.5 | 11999.5 | 0.100000 | 2.194 | -0.498 | 13 |
| 0 | ASH_COATED_OSMIUM | 10003.0 | 10008.0 | 0.000157 | 5.641 | -0.506 | 16 |
| 0 | INTARIAN_PEPPER_ROOT | 11998.5 | 13000.0 | 0.099998 | 2.364 | -0.489 | 18 |
| 1 | ASH_COATED_OSMIUM | 10008.0 | 9993.0 | -0.000713 | 4.578 | -0.491 | 22 |
| 1 | INTARIAN_PEPPER_ROOT | 13000.0 | 13999.5 | 0.100004 | 2.543 | -0.508 | 16 |

Pepper root has an OLS slope almost exactly `0.001` per timestamp, or `0.1` per 100 timestamps, on every day. Osmium has near-zero trend and residual standard deviation around 4.5 to 5.6 XIRECs, which is a much smaller and more mean-reverting process.


## Contingent-claims framing

I treat each filled unit as a short-horizon claim on terminal marked value. For a long unit, payoff is approximately `terminal_mid - fill_price`; for a short unit, payoff is `fill_price - terminal_mid`. This makes the pepper opportunity almost option-like: early inventory owns the deterministic drift while the maximum loss is limited by the 80-unit position cap and stop-loss gates.

| product | min long payoff | mean long payoff | max long payoff | interpretation |
| --- | --- | --- | --- | --- |
| ASH_COATED_OSMIUM | -23.0 | -8.0 | 2.0 | mean-reversion claim |
| INTARIAN_PEPPER_ROOT | 990.5 | 992.3 | 994.0 | trend claim |

A first-ask pepper long pays between 990.5 and 994.0 XIRECs per unit in the historical days. A first-ask osmium long is not attractive by itself, so osmium should only trade when the book is cheap or rich versus a fair value estimate, not as a directional hold.


## Statistical and ML screens

I trained simple next-tick models on days -1 and 0, then held out day 1. Features were EMA deviation, best-level imbalance, spread, and normalized timestamp. Two families were tested:

- Ridge-linear regression: stable, transparent, cheap enough to translate into live logic.
- K-nearest neighbors: useful as a nonlinear sanity check, but not deployed because the contest runtime benefits from deterministic, low-state rules and only three days of history are available.

| product | ridge MSE | baseline MSE | ridge direction | KNN MSE | KNN direction |
| --- | --- | --- | --- | --- | --- |
| ASH_COATED_OSMIUM | 7.219 | 8.571 | 69.6% | 7.641 | 67.7% |
| INTARIAN_PEPPER_ROOT | 6.797 | 8.515 | 72.0% | 7.295 | 70.4% |

Both ridge and KNN beat the mean-delta baseline on day 1, and both show directional accuracy well above 50%. The deployed live strategy uses the robust ingredients from these screens: EMA-style fair smoothing, order-book imbalance, and inventory skew. It does not ship a trained KNN table because that would add overfit surface area without improving the core pepper drift thesis.


## Strategy logic

`INTARIAN_PEPPER_ROOT`:

1. Estimate the live intercept from volume-weighted top-of-book fair value minus `0.001 * timestamp`.
2. Maintain fair value as `intercept + 0.001 * timestamp`.
3. Buy asks up to `fair + 8`, capped by `max_take = 10`, until the 80-unit limit is reached.
4. Place passive bids slightly below fair, skewed upward while inventory is below target.
5. If the observed intercept falls more than 35 XIRECs below the day-open intercept, stop adding risk and flatten with a wide emergency edge.

`ASH_COATED_OSMIUM`:

1. Estimate fair value from volume-weighted best bid/ask, blended with recent trades when present.
2. Smooth fair with `fair_alpha = 0.12` and add `0.8 * imbalance`.
3. Cross asks below fair and bids above fair with zero extra edge because the fair estimate already includes spread and imbalance filtering.
4. Add symmetric passive quotes around fair with inventory skew to avoid getting trapped at a limit.
5. If book fair moves more than 35 XIRECs from the 10,000 anchor, stop fresh mean-reversion entries and flatten inventory.


## Backtest result

| day | total PnL | pepper PnL | osmium PnL | end pepper | end osmium |
| --- | --- | --- | --- | --- | --- |
| -1 | 82493.0 | 79427.0 | 3066.0 | 80 | 75 |
| 0 | 83728.0 | 79399.0 | 4329.0 | 80 | 69 |
| 1 | 82452.0 | 79364.0 | 3088.0 | 80 | 80 |

Combined deterministic replay: **248673.0 XIRECs**. Improvement versus archived round 1 parameters: **1992.0 XIRECs**. The strategy intentionally ends long pepper because the terminal mark captures the deterministic drift. Osmium may end with inventory, but the PnL contribution is small enough that position limits and stop-loss gates dominate the risk budget.


## Risk and stress testing

Monte Carlo execution stress uses cross-fill probability `0.97`, up to `1` XIREC adverse slippage, and `3.0` XIRECs of closing mark noise. Across `40` draws, the PnL summary is:

| min | p05 | median | mean | p95 | max | Pr >= 245k |
| --- | --- | --- | --- | --- | --- | --- |
| 243993.3 | 244844.4 | 245902.9 | 245900.2 | 247099.2 | 247506.7 | 85.0% |

The main risk is a pepper regime break. The guard is intentionally model-based rather than price-based: it compares live intercept against the day-open intercept, so normal intraday timestamp drift does not trip the stop.


In [ ]:
# Re-run the notebook cells after changing trader.py.
# This repo stores the latest generated summary at logs/round2_diagnostics.json.
diagnostics['selected_parameters']
